# 🤖 Arbitbot - Multi-Exchange Crypto Arbitrage Detection Tool

This is an interactive Jupyter Notebook for real-time monitoring and detection of cryptocurrency arbitrage opportunities across multiple exchanges.


## 1️⃣ Install Dependencies and Imports


In [1]:
# Install required packages
import sys
import subprocess

packages = ['ccxt', 'ipywidgets', 'pandas', 'plotly', 'requests', 'python-telegram-bot']
for package in packages:
    try:
        __import__(package)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', package, '-q'])

print('✓ All dependencies installed')


✓ All dependencies installed


In [2]:
# Import necessary libraries
import ccxt
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
import time
from datetime import datetime
import requests
from typing import Dict, List, Optional, Tuple
from concurrent.futures import ThreadPoolExecutor, as_completed
import threading

print('✓ All libraries imported')


✓ All libraries imported


## 2️⃣ Core Functions


In [3]:
# Define cryptocurrency list - support multiple exchanges and trading pairs
CRYPTO_PAIRS = [
    {
        "name": "Bitcoin (BTC)", 
        "symbol": "BTC", 
        "pairs": {
            "binance": "BTC/USDT", 
            "bybit": "BTC/USDT", 
            "okx": "BTC/USDT", 
            "kucoin": "BTC/USDT", 
            "huobi": "btcusdt", 
            "gate": "BTC_USDT",
            "kraken": "XXBTZUSD",
            "upbit": "BTC/KRW"
        }
    },
    {
        "name": "Ethereum (ETH)", 
        "symbol": "ETH", 
        "pairs": {
            "binance": "ETH/USDT", 
            "bybit": "ETH/USDT", 
            "okx": "ETH/USDT", 
            "kucoin": "ETH/USDT", 
            "huobi": "ethusdt", 
            "gate": "ETH_USDT",
            "kraken": "XETHZUSD",
            "upbit": "ETH/KRW"
        }
    },
    {
        "name": "Ripple (XRP)", 
        "symbol": "XRP", 
        "pairs": {
            "binance": "XRP/USDT", 
            "bybit": "XRP/USDT", 
            "okx": "XRP/USDT", 
            "kucoin": "XRP/USDT", 
            "huobi": "xrpusdt", 
            "gate": "XRP_USDT",
            "kraken": "XXRPZUSD",
            "upbit": "XRP/KRW"
        }
    },
    {
        "name": "Cardano (ADA)", 
        "symbol": "ADA", 
        "pairs": {
            "binance": "ADA/USDT", 
            "bybit": "ADA/USDT", 
            "okx": "ADA/USDT", 
            "kucoin": "ADA/USDT", 
            "huobi": "adausdt", 
            "gate": "ADA_USDT",
            "kraken": "ADAUSD",
            "upbit": "ADA/KRW"
        }
    },
    {
        "name": "Solana (SOL)", 
        "symbol": "SOL", 
        "pairs": {
            "binance": "SOL/USDT", 
            "bybit": "SOL/USDT", 
            "okx": "SOL/USDT", 
            "kucoin": "SOL/USDT", 
            "huobi": "solusdt", 
            "gate": "SOL_USDT",
            "kraken": "SOLUSD",
            "upbit": "SOL/KRW"
        }
    },
    {
        "name": "Polkadot (DOT)", 
        "symbol": "DOT", 
        "pairs": {
            "binance": "DOT/USDT", 
            "bybit": "DOT/USDT", 
            "okx": "DOT/USDT", 
            "kucoin": "DOT/USDT", 
            "huobi": "dotusdt", 
            "gate": "DOT_USDT",
            "kraken": "DOTUSD",
            "upbit": "DOT/KRW"
        }
    },
    {
        "name": "Litecoin (LTC)", 
        "symbol": "LTC", 
        "pairs": {
            "binance": "LTC/USDT", 
            "bybit": "LTC/USDT", 
            "okx": "LTC/USDT", 
            "kucoin": "LTC/USDT", 
            "huobi": "ltcusdt", 
            "gate": "LTC_USDT",
            "kraken": "XLTCZUSD",
            "upbit": "LTC/KRW"
        }
    },
    {
        "name": "Bitcoin Cash (BCH)", 
        "symbol": "BCH", 
        "pairs": {
            "binance": "BCH/USDT", 
            "bybit": "BCH/USDT", 
            "okx": "BCH/USDT", 
            "kucoin": "BCH/USDT", 
            "huobi": "bchusdt", 
            "gate": "BCH_USDT",
            "kraken": "BCHUSDT",
            "upbit": "BCH/KRW"
        }
    },
    {
        "name": "Chainlink (LINK)", 
        "symbol": "LINK", 
        "pairs": {
            "binance": "LINK/USDT", 
            "bybit": "LINK/USDT", 
            "okx": "LINK/USDT", 
            "kucoin": "LINK/USDT", 
            "huobi": "linkusdt", 
            "gate": "LINK_USDT",
            "kraken": "LINKUSD",
            "upbit": "LINK/KRW"
        }
    },
    {
        "name": "UNUS SED LEO (LEO)", 
        "symbol": "LEO", 
        "pairs": {
            "binance": "LEO/USDT", 
            "bybit": "LEO/USDT", 
            "okx": "LEO/USDT", 
            "kucoin": "LEO/USDT", 
            "huobi": "leousdt", 
            "gate": "LEO_USDT",
            "kraken": "LEOUSD",
            "upbit": "LEO/KRW"
        }
    },
]

class MultiExchangeArbitrageDetector:
    """Multi-exchange arbitrage detector - support complete pair matching"""
    
    def __init__(self):
        self.exchanges = {}
        self.opportunities = []
        self.lock = threading.Lock()
    
    def initialize_exchanges(self, exchange_names: List[str]):
        """Initialize exchanges (using CCXT)"""
        self.exchanges = {}
        for name in exchange_names:
            try:
                exchange_class = getattr(ccxt, name.lower())
                self.exchanges[name] = exchange_class()
                print(f"✓ {name} connected")
            except AttributeError:
                print(f"✗ Exchange {name} not found (ensure ccxt supports it)")
            except Exception as e:
                print(f"✗ {name} connection failed: {e}")
    
    def get_price_for_pair(self, exchange_name: str, symbol: str) -> Optional[Dict]:
        """Get trading pair price from a single exchange"""
        if exchange_name not in self.exchanges:
            return None
        
        try:
            exchange = self.exchanges[exchange_name]
            ticker = exchange.fetch_ticker(symbol)
            return {
                'exchange': exchange_name,
                'symbol': symbol,
                'bid': ticker.get('bid'),
                'ask': ticker.get('ask'),
                'timestamp': datetime.now()
            }
        except Exception as e:
            return None
    
    def get_crypto_all_prices(self, crypto: Dict, exchanges: List[str]) -> Dict:
        """Concurrently fetch prices for one cryptocurrency across multiple exchanges"""
        prices = {}
        
        with ThreadPoolExecutor(max_workers=min(len(exchanges), 5)) as executor:
            futures = {}
            for ex in exchanges:
                if ex in crypto['pairs']:
                    symbol = crypto['pairs'][ex]
                    futures[executor.submit(self.get_price_for_pair, ex, symbol)] = ex
            
            for future in as_completed(futures):
                ex = futures[future]
                try:
                    price = future.result()
                    if price:
                        prices[ex] = price
                except:
                    pass
        
        return {
            'name': crypto['name'],
            'symbol': crypto['symbol'],
            'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
            'prices': prices
        }
    
    def find_all_arbitrage_pairs(self, price_data: Dict) -> List[Dict]:
        """Find all possible arbitrage pairs (A->B, B->C, etc.)"""
        if not price_data or not price_data['prices']:
            return []
        
        opportunities = []
        exchanges_list = list(price_data['prices'].keys())
        
        # Traverse all exchange combinations
        for i, buy_ex in enumerate(exchanges_list):
            for sell_ex in exchanges_list:
                if buy_ex == sell_ex:
                    continue
                
                buy_data = price_data['prices'][buy_ex]
                sell_data = price_data['prices'][sell_ex]
                
                if not buy_data or not sell_data or not buy_data.get('ask') or not sell_data.get('bid'):
                    continue
                
                buy_price = buy_data['ask']  # Use ask price for buying
                sell_price = sell_data['bid']  # Use bid price for selling
                
                profit = sell_price - buy_price
                profit_percent = (profit / buy_price) * 100 if buy_price > 0 else 0
                
                opportunities.append({
                    'symbol': price_data['symbol'],
                    'buy_exchange': buy_ex,
                    'buy_price': buy_price,
                    'sell_exchange': sell_ex,
                    'sell_price': sell_price,
                    'profit': profit,
                    'profit_percent': profit_percent,
                    'timestamp': price_data['timestamp']
                })
        
        return opportunities

# Initialize detector
detector = MultiExchangeArbitrageDetector()
print('✓ MultiExchangeArbitrageDetector initialized (complete pair matching mode)')


✓ MultiExchangeArbitrageDetector initialized (complete pair matching mode)


In [4]:
def send_telegram(message: str, token: str, chat_id: str) -> bool:
    """Send Telegram message"""
    if not token or not chat_id:
        print('⚠ Telegram configuration incomplete')
        return False
    
    try:
        url = f'https://api.telegram.org/bot{token}/sendMessage'
        payload = {
            'chat_id': chat_id,
            'text': message,
            'parse_mode': 'HTML'
        }
        response = requests.post(url, json=payload, timeout=10)
        return response.status_code == 200
    except Exception as e:
        print(f'✗ Telegram send failed: {e}')
        return False

print('✓ Telegram function defined')


✓ Telegram function defined


## 3️⃣ GUI Setup


In [ ]:
# ============ GUI Component Definition ============

# Title
title = widgets.HTML('<h2 style="color:#2E86AB; text-align:center;">🤖 Multi-Exchange Arbitrage Detection Tool</h2>')

# Exchange Selection - Dual list (left selectable, right display)
exchange_selector = widgets.SelectMultiple(
    options=['binance', 'bybit', 'okx', 'kucoin', 'huobi', 'gate', 'kraken', 'upbit', 'coinbase', 'bitfinex'],
    value=['binance', 'bybit', 'okx'],
    description='Available Exchanges:',
    style={'description_width': '130px'},
    layout=widgets.Layout(width='350px', height='150px')
)

exchange_display = widgets.Textarea(
    value='binance, bybit, okx',
    placeholder='Selected exchanges will display here',
    description='Selected:',
    style={'description_width': '130px'},
    disabled=True,
    layout=widgets.Layout(width='350px', height='150px')
)

# Cryptocurrency Selection - Dual list (left selectable, right display)
crypto_symbols = [f"{c['symbol']} - {c['name']}" for c in CRYPTO_PAIRS]
crypto_selector = widgets.SelectMultiple(
    options=crypto_symbols,
    value=[crypto_symbols[0], crypto_symbols[1], crypto_symbols[2]],  # BTC, ETH, XRP
    description='Available Cryptos:',
    style={'description_width': '130px'},
    layout=widgets.Layout(width='350px', height='150px')
)

crypto_display = widgets.Textarea(
    value='\n'.join([crypto_symbols[0], crypto_symbols[1], crypto_symbols[2]]),
    placeholder='Selected cryptos will display here',
    description='Selected:',
    style={'description_width': '130px'},
    disabled=True,
    layout=widgets.Layout(width='350px', height='150px')
)

# Profit Threshold - precision to 0.01
profit_threshold = widgets.FloatSlider(
    value=0.5,
    min=0.01,
    max=5.0,
    step=0.01,
    description='Profit Threshold (%):',
    style={'description_width': '130px'},
    layout=widgets.Layout(width='700px')
)

# Check Interval - add 5 min, 30 min, 1 hour
check_interval = widgets.Dropdown(
    options={
        '3 seconds': 3,
        '5 seconds': 5,
        '10 seconds': 10,
        '30 seconds': 30,
        '60 seconds': 60,
        '5 minutes': 300,
        '30 minutes': 1800,
        '1 hour': 3600,
    },
    value=10,
    description='Check Interval:',
    style={'description_width': '130px'},
    layout=widgets.Layout(width='700px')
)

# Telegram Configuration
tg_token_input = widgets.Password(
    placeholder='Enter Telegram Bot Token',
    description='TG Token:',
    style={'description_width': '130px'},
    layout=widgets.Layout(width='700px')
)

tg_chat_id_input = widgets.Text(
    placeholder='Enter Chat ID',
    description='TG Chat ID:',
    style={'description_width': '130px'},
    layout=widgets.Layout(width='700px')
)

# Telegram Enable Toggle
enable_telegram = widgets.Checkbox(
    value=False,
    description='Enable Telegram Push',
    indent=False,
    layout=widgets.Layout(width='300px')
)

# Run Button
run_button = widgets.Button(
    description='🚀 Start Detection',
    button_style='success',
    layout=widgets.Layout(width='200px', height='40px')
)

# Stop Button
stop_button = widgets.Button(
    description='⏹ Stop',
    button_style='danger',
    layout=widgets.Layout(width='200px', height='40px')
)

# Result Output
output_area = widgets.Output(layout=widgets.Layout(width='100%', border='1px solid #ccc', padding='10px', height='300px'))

# Table Output
table_output = widgets.Output(layout=widgets.Layout(width='100%', border='1px solid #ccc', padding='10px', height='400px'))

# Orderbook Output (new) - for displaying bid/ask for each crypto
orderbook_output = widgets.Output(layout=widgets.Layout(width='100%', border='1px solid #2E86AB', padding='10px', height='300px', background_color='#f0f8ff'))

# Sync display of selected exchanges
def update_exchange_display(change):
    exchange_display.value = ', '.join(exchange_selector.value) if exchange_selector.value else '(None selected)'

exchange_selector.observe(update_exchange_display, names='value')

# Sync display of selected cryptocurrencies
def update_crypto_display(change):
    crypto_display.value = '\n'.join(crypto_selector.value) if crypto_selector.value else '(None selected)'

crypto_selector.observe(update_crypto_display, names='value')

print('✓ GUI components defined (wider layout + orderbook monitor)')


✓ GUI components defined (wider layout + orderbook monitor)


## 4️⃣ Launch GUI


In [7]:
# Organize GUI layout
config_box = widgets.VBox([
    widgets.HTML('<h3>⚙️ Configuration</h3>'),
    widgets.HTML('<b>Select Exchanges (left: select, right: display):</b>'),
    widgets.HBox([exchange_selector, exchange_display]),
    widgets.HTML('<b>Select Cryptos (left: select, right: display):</b>'),
    widgets.HBox([crypto_selector, crypto_display]),
    widgets.HTML('<b>Detection Settings:</b>'),
    profit_threshold,
    check_interval,
])

tg_box = widgets.VBox([
    widgets.HTML('<h3>📱 Telegram Configuration</h3>'),
    enable_telegram,
    tg_token_input,
    tg_chat_id_input,
    widgets.HTML('<p style="font-size:12px; color:#999;">💡 Tip: Request Bot Token from @BotFather on Telegram</p>')
])

button_box = widgets.HBox([run_button, stop_button])

main_layout = widgets.VBox([
    title,
    widgets.HTML('<hr style="margin: 20px 0;">'),
    config_box,
    widgets.HTML('<hr style="margin: 20px 0;">'),
    tg_box,
    widgets.HTML('<hr style="margin: 20px 0;">'),
    button_box,
    widgets.HTML('<h3>📊 Real-time Results</h3>'),
    output_area,
    widgets.HTML('<h3>📈 Arbitrage Opportunities</h3>'),
    table_output,
    widgets.HTML('<h3>📖 Orderbook (Bid/Ask Snapshot)</h3>'),
    orderbook_output,
])

display(main_layout)


In [3]:
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import pandas as pd
from datetime import datetime
import time
import requests 
from concurrent.futures import ThreadPoolExecutor, as_completed 
import ccxt
import sys
import matplotlib.pyplot as plt 
import random 

# ----------------------------------------------------
# 📌 終端字體設定 (根據您的作業系統需求)
# 我的作業系統是 ubuntu 24.04.02，這裡使用「文泉驛微米黑」作為中文字體。
# 如果程式碼中任何部分使用了 matplotlib.pyplot (plt) 進行繪圖，這將確保中文字體顯示正常。
plt.rcParams['font.sans-serif'] = ['WenQuanYi Micro Hei']
plt.rcParams['axes.unicode_minus'] = False
# ----------------------------------------------------

# --- Config and Utilities ---
# 修正：將所有 CCXT 支援的活躍幣種都拉回清單
CRYPTO_PAIRS = [
    {'symbol': 'BTC/USDT', 'name': 'Bitcoin'},
    {'symbol': 'ETH/USDT', 'name': 'Ethereum'},
    {'symbol': 'XRP/USDT', 'name': 'Ripple'},
    {'symbol': 'DOGE/USDT', 'name': 'Dogecoin'}, 
    {'symbol': 'SOL/USDT', 'name': 'Solana'},
    {'symbol': 'ADA/USDT', 'name': 'Cardano'}, 
    {'symbol': 'DOT/USDT', 'name': 'Polkadot'},
    {'symbol': 'LTC/USDT', 'name': 'Litecoin'},
    {'symbol': 'BCH/USDT', 'name': 'Bitcoin Cash'},
    {'symbol': 'LINK/USDT', 'name': 'Chainlink'}, 
    {'symbol': 'VET/USDT', 'name': 'VeChain'}, 
    {'symbol': 'TRX/USDT', 'name': 'TRON'}, 
    {'symbol': 'MATIC/USDT', 'name': 'Polygon'},
    {'symbol': 'AVAX/USDT', 'name': 'Avalanche'},
] 

# CCXT 支援所有這些交易所。
ALL_EXCHANGES = ['binance', 'bybit', 'okx', 'kucoin', 'huobi', 'gate', 'kraken', 'coinbase', 'bitfinex']

class ArbitrageDetector:
    def __init__(self):
        self.fees = {} 
        self.clients = {}
        self.unified_symbols = {c['symbol']: c['name'] for c in CRYPTO_PAIRS}

    def initialize_exchanges(self, exchanges):
        """初始化 CCXT 交易所客戶端並啟用速率限制。"""
        self.clients = {}
        timeout_ms = 10000 
        for exchange_id in exchanges:
            try:
                exchange_class = getattr(ccxt, exchange_id)
                self.clients[exchange_id] = exchange_class({
                    'timeout': timeout_ms,
                    'enableRateLimit': True, # 啟用內建速率限制
                })
            except Exception as e:
                pass

    def get_crypto_all_prices(self, crypto, exchanges):
        """使用 CCXT 獲取指定交易對在各交易所的真實買一價和賣一價。"""
        prices = {}
        unified_symbol = crypto['symbol']
        
        for exchange_id in exchanges:
            exchange = self.clients.get(exchange_id)
            if not exchange: 
                continue

            try:
                ticker = exchange.fetch_ticker(unified_symbol)
                
                bid_price = ticker.get('bid') 
                ask_price = ticker.get('ask')

                if bid_price is not None and ask_price is not None:
                    prices[exchange_id] = {
                        'bid': bid_price,
                        'ask': ask_price,
                    }
                
            except ccxt.ExchangeNotAvailable as e:
                pass 
            except ccxt.DDoSProtection or ccxt.RateLimitExceeded as e:
                pass
            except Exception as e:
                pass 
                
        return {
            'symbol': unified_symbol,
            'name': crypto['name'],
            'prices': prices
        }

    def find_all_arbitrage_pairs(self, price_data):
        opportunities = []
        symbol = price_data['symbol']
        exchanges = list(price_data['prices'].keys())
        
        for i in range(len(exchanges)):
            for j in range(len(exchanges)):
                if i == j:
                    continue
                
                buy_ex = exchanges[i]
                sell_ex = exchanges[j]
                
                buy_price = price_data['prices'][buy_ex].get('ask') 
                sell_price = price_data['prices'][sell_ex].get('bid') 
                
                buy_fee = self.fees.get(buy_ex, 0)
                sell_fee = self.fees.get(sell_ex, 0)
                
                if buy_price and sell_price and buy_price > 0:
                    
                    net_sell_price = sell_price * (1 - sell_fee)
                    net_profit_ratio = (net_sell_price - buy_price) / buy_price
                    profit_percent = net_profit_ratio * 100
                    
                    if profit_percent >= 0.001: 
                        opportunities.append({
                            'symbol': symbol,
                            'buy_exchange': buy_ex,
                            'sell_exchange': sell_ex,
                            'buy_price': buy_price,
                            'sell_price': sell_price,
                            'buy_fee': buy_fee, 
                            'sell_fee': sell_fee,
                            'net_sell_price': net_sell_price, 
                            'profit_percent': profit_percent,
                        })
        return opportunities

detector = ArbitrageDetector() 

# --- UI Component Definition ---

## 🏆 Title
title = widgets.HTML(
    '<div style="background-color:#6A5ACD; color:white; padding:10px; border-radius:8px; text-align:center;">'
    '<h2>🚀 Cross-Exchange Arbitrage Detector (CCXT)</h2>'
    '</div>'
)

## 1️⃣ Exchange Selection
exchange_selector = widgets.SelectMultiple(
    options=ALL_EXCHANGES,
    value=['binance', 'bybit', 'okx'],
    description='',
    layout=widgets.Layout(width='150px', height='250px') 
)

exchange_display = widgets.Textarea(
    value='binance, bybit, okx',
    placeholder='Selected...',
    disabled=True,
    layout=widgets.Layout(width='180px', height='250px') 
)

exchange_button = widgets.Button(
    description='✅ Confirm',
    button_style='success',
    layout=widgets.Layout(width='150px', height='30px') 
)

def update_exchange_display(b):
    exchange_display.value = ', '.join(exchange_selector.value) if exchange_selector.value else '(None)'
exchange_button.on_click(update_exchange_display)

exchange_box = widgets.VBox([
    widgets.HTML('<b style="font-size:14px;">1️⃣ Exchanges</b>'),
    widgets.HBox([
        widgets.VBox([widgets.HTML('<span style="font-size:12px;">Available:</span>'), exchange_selector]),
        widgets.VBox([widgets.HTML('<span style="font-size:12px;">Selected:</span>'), exchange_display]),
    ], layout=widgets.Layout(gap='10px')),
    exchange_button
])


## 2️⃣ Crypto Selection
crypto_symbols = [f"{c['symbol']} - {c['name']}" for c in CRYPTO_PAIRS]
crypto_selector = widgets.SelectMultiple(
    options=crypto_symbols,
    # 預設選取前三個幣種
    value=[crypto_symbols[0], crypto_symbols[1], crypto_symbols[2]] if len(crypto_symbols) >= 3 else crypto_symbols,
    description='',
    layout=widgets.Layout(width='200px', height='250px') 
)

selected_initial_symbols = [text.split(' - ')[0] for text in crypto_selector.value]
crypto_display = widgets.Textarea(
    value='\n'.join(selected_initial_symbols),
    placeholder='Selected...',
    disabled=True,
    layout=widgets.Layout(width='120px', height='250px') 
)

crypto_button = widgets.Button(
    description='✅ Confirm',
    button_style='success',
    layout=widgets.Layout(width='150px', height='30px') 
)

def update_crypto_display(b):
    selected_symbols = [text.split(' - ')[0] for text in crypto_selector.value]
    crypto_display.value = '\n'.join(selected_symbols) if selected_symbols else '(None)'
crypto_button.on_click(update_crypto_display)

crypto_box = widgets.VBox([
    widgets.HTML('<b style="font-size:14px;">2️⃣ Cryptos</b>'),
    widgets.HBox([
        widgets.VBox([widgets.HTML('<span style="font-size:12px;">Available:</span>'), crypto_selector]),
        widgets.VBox([widgets.HTML('<span style="font-size:12px;">Selected:</span>'), crypto_display]),
    ], layout=widgets.Layout(gap='10px')),
    crypto_button
])

# 第一行：Exchange + Crypto
row1 = widgets.HBox([exchange_box, crypto_box], layout=widgets.Layout(gap='50px', margin='5px 0'))


## 🔧 Taker Fees
fee_widgets = {}
# 假設費率皆為 Taker Fee，且與交易所的預設費率保持一致
initial_fees = {ex: 0.001 for ex in ALL_EXCHANGES}
initial_fees['bybit'] = 0.0007 
initial_fees['okx'] = 0.0008

for ex in ALL_EXCHANGES:
    fee_widgets[ex] = widgets.BoundedFloatText(
        value=initial_fees.get(ex, 0.001), 
        min=0.0,
        max=0.01,
        step=0.0001,
        description=ex.capitalize(),
        style={'description_width': '80px'},
        layout=widgets.Layout(width='160px')
    )

fee_row1 = widgets.HBox([fee_widgets[ex] for ex in ALL_EXCHANGES[:5]], layout=widgets.Layout(gap='15px'))
fee_row2 = widgets.HBox([fee_widgets[ex] for ex in ALL_EXCHANGES[5:]], layout=widgets.Layout(gap='15px'))

fee_box = widgets.VBox([
    widgets.HTML('<b style="font-size:14px;">3️⃣ Taker Fees:</b>'),
    fee_row1,
    fee_row2
], layout=widgets.Layout(border='1px solid #555', padding='10px', margin='10px 0'))


## 4️⃣ Parameters & Telegram
profit_threshold = widgets.FloatSlider(
    value=0.5,
    min=0.01,
    max=5.0,
    step=0.01,
    description='Min Profit %',
    style={'description_width': '100px'},
    readout_format='.2f', 
    layout=widgets.Layout(width='350px')
)

# 修正：刪除 3s, 5s, 10s 選項
check_interval = widgets.Dropdown(
    options={'30s': 30, '1min': 60, '5min': 300, '10min': 600, '30min': 1800, '1hr': 3600},
    value=30, # 更新預設值
    description='Interval',
    style={'description_width': '100px'},
    layout=widgets.Layout(width='220px')
)

enable_telegram = widgets.Checkbox(
    value=False,
    description='Enable TG Notifications',
    indent=False,
    layout=widgets.Layout(width='200px')
)

tg_token_input = widgets.Password(
    placeholder='Telegram Bot Token',
    description='Token:',
    style={'description_width': '60px'},
    layout=widgets.Layout(width='300px')
)

tg_chat_id_input = widgets.Text(
    placeholder='Telegram Chat ID',
    description='Chat ID:',
    style={'description_width': '60px'},
    layout=widgets.Layout(width='200px')
)

settings_box = widgets.HBox([
    widgets.VBox([
        widgets.HTML('<b style="font-size:14px;">4️⃣ Parameters</b>'),
        widgets.HBox([profit_threshold, check_interval], layout=widgets.Layout(gap='30px'))
    ]),
    widgets.VBox([
        widgets.HTML('<b style="font-size:14px;">5️⃣ Telegram</b>'),
        widgets.HBox([enable_telegram], layout=widgets.Layout(gap='10px')),
        widgets.HBox([tg_token_input, tg_chat_id_input], layout=widgets.Layout(gap='10px'))
    ])
], layout=widgets.Layout(gap='60px', margin='10px 0'))


## ▶️ Run Button
run_button = widgets.Button(
    description='🚀 Start Detection',
    button_style='success',
    layout=widgets.Layout(width='180px', height='40px')
)

button_box = widgets.HBox([run_button], layout=widgets.Layout(justify_content='center', margin='15px 0'))


## 🖥️ Output Areas 
# 📌 修正：表格顯示 Top 10 (高度調整為 500px)
table_output = widgets.Output(layout=widgets.Layout(width='100%', border='2px solid #3CB371', padding='15px', height='500px', overflow_y='auto')) 
# 📌 修正：Log 區域空間維持 1000px
output_area = widgets.Output(layout=widgets.Layout(width='100%', border='2px solid #999', padding='15px', height='1000px', overflow_y='auto')) 


# --- Main Layout ---
main_layout = widgets.VBox([
    title,
    widgets.HTML('<hr style="margin: 10px 0; border: 1px solid #ddd;">'),
    row1,
    widgets.HTML('<hr style="margin: 10px 0; border: 1px solid #ddd;">'),
    fee_box,
    widgets.HTML('<hr style="margin: 10px 0; border: 1px solid #ddd;">'),
    settings_box,
    widgets.HTML('<hr style="margin: 10px 0; border: 1px solid #ddd;">'),
    button_box,
    widgets.HTML('<b>📈 Arbitrage Opportunities (Top 10)</b>'),
    table_output,
    widgets.HTML('<b>💻 Log / Console Output</b>'),
    output_area,
], layout=widgets.Layout(width='100%')) 


# --- Event Handler Logic ---

def send_telegram_notification(token, chat_id, msg):
    if not token or not chat_id: return 
    try:
        url = f'https://api.telegram.org/bot{token}/sendMessage'
        payload = {'chat_id': chat_id, 'text': msg, 'parse_mode': 'HTML'}
        requests.post(url, json=payload, timeout=5)
    except Exception as e:
        with output_area: print(f'❌ TG failed: {str(e)[:40]}')

def get_current_fees():
    return {ex: fee_widgets[ex].value for ex in ALL_EXCHANGES}

running = False

def on_run_clicked(b):
    global running, detector
    
    if running:
        with output_area: clear_output(); print('⚠️ Detection is already running. Please interrupt the kernel to stop.')
        return

    running = True
    run_button.disabled = True
    
    # 獲取配置參數
    exchanges_str = exchange_display.value
    cryptos_symbols_str = crypto_display.value
    
    exchanges = [e.strip() for e in exchanges_str.split(',') if e.strip() and e.strip() != '(None)']
    selected_symbols = [s.strip() for s in cryptos_symbols_str.split('\n') if s.strip() and s.strip() != '(None)']
    
    # 構建選中的 Crypto 列表 (包含 symbol 和 name)
    selected_cryptos = [c for c in CRYPTO_PAIRS if c['symbol'] in selected_symbols]

    
    threshold = profit_threshold.value
    interval = check_interval.value
    enable_tg = enable_telegram.value
    tg_token = tg_token_input.value if enable_tg else ''
    tg_chat_id = tg_chat_id_input.value if enable_tg else ''
    
    # 檢查配置
    if not exchanges:
        with output_area: clear_output(); print('❌ Select exchanges first')
        running = False; run_button.disabled = False; return
    if not selected_cryptos:
        with output_area: clear_output(); print('❌ Select cryptos first')
        running = False; run_button.disabled = False; return
    if enable_tg and (not tg_token or not tg_chat_id):
        with output_area: clear_output(); print('❌ TG config incomplete')
        running = False; run_button.disabled = False; return
    
    # 初始化 CCXT 客戶端和費用
    detector.fees = get_current_fees()
    detector.initialize_exchanges(exchanges) # 初始化 CCXT 客戶端
    
    with output_area:
        clear_output()
        print('--- Arbitrage Detector Initialized ---')
        print(f'🚨 Stop: interrupt Jupyter kernel')
        print(f'Exchanges: {len(exchanges)} | Cryptos: {len(selected_cryptos)}')
        print(f'Min Profit: {threshold}% | Interval: {interval}s')
        print(f'Telegram: {"Enabled" if enable_tg else "Disabled"}')
        print('------------------------------------')
    
    
    iteration = 0
    while running:
        iteration += 1
        round_opportunities = []
        
        with output_area:
            clear_output(wait=True)
            print(f'🔄 #{iteration} | {datetime.now().strftime("%Y-%m-%d %H:%M:%S")} | Fetching Prices via CCXT...')

        if not running: break

        # 使用線程池並發獲取價格
        with ThreadPoolExecutor(max_workers=len(selected_cryptos) * 2) as executor:
            # 傳遞完整的 crypto 物件和選中的交易所列表
            futures = {executor.submit(detector.get_crypto_all_prices, crypto, exchanges): crypto for crypto in selected_cryptos}
            
            for future in as_completed(futures):
                if not running: break
                crypto = futures[future]
                unified_symbol = crypto['symbol']
                
                try:
                    price_data = future.result()
                    
                    # Log 顯示 CCXT 抓到的數據價格
                    if price_data.get('prices'):
                        log_msg = f"🔎 {unified_symbol} Prices:"
                        # 顯示交易所簡稱 (前 4 個字元) 及 Ask/Bid 價格
                        for ex, p in price_data['prices'].items():
                            ask = f"{p['ask']:.6f}" if p['ask'] is not None else 'N/A'
                            bid = f"{p['bid']:.6f}" if p['bid'] is not None else 'N/A'
                            log_msg += f" {ex[:4]}={ask}/{bid}"
                        with output_area: print(log_msg)
                    
                    # 檢查是否收到有效數據
                    if not price_data or not price_data.get('prices') or len(price_data['prices']) < 2: # 至少需要兩個價格才能套利
                        with output_area: print(f'⚠️ {unified_symbol}: Not enough valid price data received (skipping calculation)')
                        continue 

                    all_pairs = detector.find_all_arbitrage_pairs(price_data) 
                    profitable_pairs = [p for p in all_pairs if p['profit_percent'] >= threshold]
                    
                    if profitable_pairs:
                        profitable_pairs.sort(key=lambda x: x['profit_percent'], reverse=True)
                        round_opportunities.extend(profitable_pairs)
                        with output_area: print(f'✅ {unified_symbol}: Found {len(profitable_pairs)} opportunities, Max: {profitable_pairs[0]["profit_percent"]:.4f}%')

                        if enable_tg:
                            best = profitable_pairs[0]
                            # 使用 HTML 格式化 Telegram 訊息
                            msg = f'🔔 <b>套利機會: {crypto["name"]} ({best["profit_percent"]:.4f}%)</b>\n買入: {best["buy_exchange"].capitalize()} @ ${best["buy_price"]:.6f}\n賣出: {best["sell_exchange"].capitalize()} @ ${best["sell_price"]:.6f}'
                            send_telegram_notification(tg_token, tg_chat_id, msg)
                    else:
                        with output_area: print(f'ℹ️ {unified_symbol}: No opportunities > {threshold}%')
                
                except Exception as e:
                    # 捕捉線程執行期間的任何意外錯誤
                    with output_area: print(f'❌ {unified_symbol} General Error: {str(e)}')
            
            if not running: break
            
        if not running: break
        
        # 更新套利機會表格
        with table_output:
            clear_output(wait=True)
            if round_opportunities:
                # 僅顯示前 10 筆
                df = pd.DataFrame(round_opportunities).sort_values('profit_percent', ascending=False).head(10)
                df_display = df[['symbol', 'buy_exchange', 'sell_exchange', 'buy_price', 'sell_price', 'buy_fee', 'sell_fee', 'profit_percent']].copy()
                
                df_display.columns = ['Crypto', 'Buy', 'Sell', 'Ask Price (Buy)', 'Bid Price (Sell)', 'Buy Fee (%)', 'Sell Fee (%)', 'Profit (%)']
                
                # 格式化數值
                df_display['Profit (%)'] = df_display['Profit (%)'].round(4)
                df_display['Buy Fee (%)'] = (df_display['Buy Fee (%)'] * 100).round(4)
                df_display['Sell Fee (%)'] = (df_display['Sell Fee (%)'] * 100).round(4)
                df_display['Ask Price (Buy)'] = df_display['Ask Price (Buy)'].round(6) 
                df_display['Bid Price (Sell)'] = df_display['Bid Price (Sell)'].round(6) 

                html_table = df_display.to_html(index=False, classes='table table-striped', float_format='%.6f')
                display(HTML(f'<b>Found Total: {len(round_opportunities)} | Displaying Top {len(df_display)}</b><br>{html_table}'))
            else:
                # 最終總結：如果 round_opportunities 為空，則輸出此訊息
                print('ℹ️ No opportunities found above the minimum profit threshold.')

        if not running: break
        time.sleep(interval)
    
    with output_area: print('⏹ Stopped.')
    run_button.disabled = False


run_button.on_click(on_run_clicked)
display(main_layout)

In [1]:
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import pandas as pd
from datetime import datetime
import time
import requests
from concurrent.futures import ThreadPoolExecutor, as_completed
import ccxt
import sys
import matplotlib.pyplot as plt
import random
import threading # 新增: 用於執行緒鎖定
from requests.exceptions import RequestException # 新增: 用於更精準的網路錯誤捕捉

# ----------------------------------------------------
# 📌 終端字體設定 (根據您的作業系統需求)
# 我的作業系統是 ubuntu 24.04.02，這裡使用「文泉驛微米黑」作為中文字體。
# 如果程式碼中任何部分使用了 matplotlib.pyplot (plt) 進行繪圖，這將確保中文字體顯示正常。
plt.rcParams['font.sans-serif'] = ['WenQuanYi Micro Hei']
plt.rcParams['axes.unicode_minus'] = False
# ----------------------------------------------------

# --- Config and Utilities ---
CRYPTO_PAIRS = [
    {'symbol': 'BTC/USDT', 'name': 'Bitcoin'},
    {'symbol': 'ETH/USDT', 'name': 'Ethereum'},
    {'symbol': 'XRP/USDT', 'name': 'Ripple'},
    {'symbol': 'DOGE/USDT', 'name': 'Dogecoin'},
    {'symbol': 'SOL/USDT', 'name': 'Solana'},
    {'symbol': 'ADA/USDT', 'name': 'Cardano'},
    {'symbol': 'DOT/USDT', 'name': 'Polkadot'},
    {'symbol': 'LTC/USDT', 'name': 'Litecoin'},
    {'symbol': 'BCH/USDT', 'name': 'Bitcoin Cash'},
    {'symbol': 'LINK/USDT', 'name': 'Chainlink'},
    {'symbol': 'VET/USDT', 'name': 'VeChain'},
    {'symbol': 'TRX/USDT', 'name': 'TRON'},
    {'symbol': 'MATIC/USDT', 'name': 'Polygon'},
    {'symbol': 'AVAX/USDT', 'name': 'Avalanche'},
]

ALL_EXCHANGES = ['binance', 'bybit', 'okx', 'kucoin', 'huobi', 'gate', 'kraken', 'coinbase', 'bitfinex']

class ArbitrageDetector:
    def __init__(self):
        self.fees = {}
        self.clients = {}
        self.unified_symbols = {c['symbol']: c['name'] for c in CRYPTO_PAIRS}
        # 修正: 新增一個用於 TG 發送的線程池，避免阻塞主循環
        self.tg_executor = ThreadPoolExecutor(max_workers=5)

    def initialize_exchanges(self, exchanges):
        """初始化 CCXT 交易所客戶端並啟用速率限制。"""
        self.clients = {}
        timeout_ms = 10000
        for exchange_id in exchanges:
            try:
                exchange_class = getattr(ccxt, exchange_id)
                self.clients[exchange_id] = exchange_class({
                    'timeout': timeout_ms,
                    'enableRateLimit': True, # 啟用內建速率限制
                })
            except Exception as e:
                pass

    def get_crypto_all_prices(self, crypto, exchanges):
        """使用 CCXT 獲取指定交易對在各交易所的真實買一價和賣一價。"""
        prices = {}
        unified_symbol = crypto['symbol']

        for exchange_id in exchanges:
            exchange = self.clients.get(exchange_id)
            if not exchange:
                continue

            try:
                ticker = exchange.fetch_ticker(unified_symbol)

                bid_price = ticker.get('bid')
                ask_price = ticker.get('ask')

                if bid_price is not None and ask_price is not None:
                    prices[exchange_id] = {
                        'bid': bid_price,
                        'ask': ask_price,
                    }

            except ccxt.ExchangeNotAvailable:
                pass
            except ccxt.DDoSProtection or ccxt.RateLimitExceeded:
                pass
            except Exception:
                pass # 其他 CCXT 錯誤

        return {
            'symbol': unified_symbol,
            'name': crypto['name'],
            'prices': prices
        }

    def find_all_arbitrage_pairs(self, price_data):
        """計算所有套利機會。"""
        opportunities = []
        symbol = price_data['symbol']
        exchanges = list(price_data['prices'].keys())

        # 修正: 確保在計算時使用 UI 上即時的費用設定 (雖然 fee_widgets 是全域變數，但這裡重新讀取更保險)
        current_fees = get_current_fees()

        for i in range(len(exchanges)):
            for j in range(len(exchanges)):
                if i == j:
                    continue

                buy_ex = exchanges[i]
                sell_ex = exchanges[j]

                buy_price = price_data['prices'][buy_ex].get('ask')
                sell_price = price_data['prices'][sell_ex].get('bid')

                buy_fee = current_fees.get(buy_ex, 0)
                sell_fee = current_fees.get(sell_ex, 0)

                if buy_price and sell_price and buy_price > 0:
                    
                    # 淨賣出價格 (考慮賣出手續費)
                    net_sell_price = sell_price * (1 - sell_fee)
                    
                    # 淨利潤率 (簡化模型：忽略買入費用對基礎貨幣的影響，僅計算買入成本與淨賣出收入的差值百分比)
                    net_profit_ratio = (net_sell_price - buy_price) / buy_price
                    profit_percent = net_profit_ratio * 100

                    # 使用 UI 設置的最小利潤門檻
                    if profit_percent >= profit_threshold.value:
                        opportunities.append({
                            'symbol': symbol,
                            'buy_exchange': buy_ex,
                            'sell_exchange': sell_ex,
                            'buy_price': buy_price,
                            'sell_price': sell_price,
                            'buy_fee': buy_fee,
                            'sell_fee': sell_fee,
                            'net_sell_price': net_sell_price,
                            'profit_percent': profit_percent,
                        })
        return opportunities

detector = ArbitrageDetector()

# --- UI Component Definition ---
# (此處保留原程式碼中所有的 UI 定義，確保 UI 不變)

## 🏆 Title
title = widgets.HTML(
    '<div style="background-color:#6A5ACD; color:white; padding:10px; border-radius:8px; text-align:center;">'
    '<h2>🚀 Cross-Exchange Arbitrage Detector (CCXT)</h2>'
    '</div>'
)

## 1️⃣ Exchange Selection
exchange_selector = widgets.SelectMultiple(
    options=ALL_EXCHANGES,
    value=['binance', 'bybit', 'okx'],
    description='',
    layout=widgets.Layout(width='150px', height='250px')
)

exchange_display = widgets.Textarea(
    value='binance, bybit, okx',
    placeholder='Selected...',
    disabled=True,
    layout=widgets.Layout(width='180px', height='250px')
)

exchange_button = widgets.Button(
    description='✅ Confirm',
    button_style='success',
    layout=widgets.Layout(width='150px', height='30px')
)

def update_exchange_display(b):
    exchange_display.value = ', '.join(exchange_selector.value) if exchange_selector.value else '(None)'
exchange_button.on_click(update_exchange_display)

exchange_box = widgets.VBox([
    widgets.HTML('<b style="font-size:14px;">1️⃣ Exchanges</b>'),
    widgets.HBox([
        widgets.VBox([widgets.HTML('<span style="font-size:12px;">Available:</span>'), exchange_selector]),
        widgets.VBox([widgets.HTML('<span style="font-size:12px;">Selected:</span>'), exchange_display]),
    ], layout=widgets.Layout(gap='10px')),
    exchange_button
])


## 2️⃣ Crypto Selection
crypto_symbols = [f"{c['symbol']} - {c['name']}" for c in CRYPTO_PAIRS]
crypto_selector = widgets.SelectMultiple(
    options=crypto_symbols,
    # 預設選取前三個幣種
    value=[crypto_symbols[0], crypto_symbols[1], crypto_symbols[2]] if len(crypto_symbols) >= 3 else crypto_symbols,
    description='',
    layout=widgets.Layout(width='200px', height='250px')
)

selected_initial_symbols = [text.split(' - ')[0] for text in crypto_selector.value]
crypto_display = widgets.Textarea(
    value='\n'.join(selected_initial_symbols),
    placeholder='Selected...',
    disabled=True,
    layout=widgets.Layout(width='120px', height='250px')
)

crypto_button = widgets.Button(
    description='✅ Confirm',
    button_style='success',
    layout=widgets.Layout(width='150px', height='30px')
)

def update_crypto_display(b):
    selected_symbols = [text.split(' - ')[0] for text in crypto_selector.value]
    crypto_display.value = '\n'.join(selected_symbols) if selected_symbols else '(None)'
crypto_button.on_click(update_crypto_display)

crypto_box = widgets.VBox([
    widgets.HTML('<b style="font-size:14px;">2️⃣ Cryptos</b>'),
    widgets.HBox([
        widgets.VBox([widgets.HTML('<span style="font-size:12px;">Available:</span>'), crypto_selector]),
        widgets.VBox([widgets.HTML('<span style="font-size:12px;">Selected:</span>'), crypto_display]),
    ], layout=widgets.Layout(gap='10px')),
    crypto_button
])

# 第一行：Exchange + Crypto
row1 = widgets.HBox([exchange_box, crypto_box], layout=widgets.Layout(gap='50px', margin='5px 0'))


## 🔧 Taker Fees
fee_widgets = {}
# 假設費率皆為 Taker Fee，且與交易所的預設費率保持一致
initial_fees = {ex: 0.001 for ex in ALL_EXCHANGES}
initial_fees['bybit'] = 0.0007
initial_fees['okx'] = 0.0008

for ex in ALL_EXCHANGES:
    fee_widgets[ex] = widgets.BoundedFloatText(
        value=initial_fees.get(ex, 0.001),
        min=0.0,
        max=0.01,
        step=0.0001,
        description=ex.capitalize(),
        style={'description_width': '80px'},
        layout=widgets.Layout(width='160px')
    )

fee_row1 = widgets.HBox([fee_widgets[ex] for ex in ALL_EXCHANGES[:5]], layout=widgets.Layout(gap='15px'))
fee_row2 = widgets.HBox([fee_widgets[ex] for ex in ALL_EXCHANGES[5:]], layout=widgets.Layout(gap='15px'))

fee_box = widgets.VBox([
    widgets.HTML('<b style="font-size:14px;">3️⃣ Taker Fees:</b>'),
    fee_row1,
    fee_row2
], layout=widgets.Layout(border='1px solid #555', padding='10px', margin='10px 0'))


## 4️⃣ Parameters & Telegram
profit_threshold = widgets.FloatSlider(
    value=0.5,
    min=0.01,
    max=5.0,
    step=0.01,
    description='Min Profit %',
    style={'description_width': '100px'},
    readout_format='.2f',
    layout=widgets.Layout(width='350px')
)

check_interval = widgets.Dropdown(
    options={'30s': 30, '1min': 60, '5min': 300, '10min': 600, '30min': 1800, '1hr': 3600},
    value=30, # 更新預設值
    description='Interval',
    style={'description_width': '100px'},
    layout=widgets.Layout(width='220px')
)

enable_telegram = widgets.Checkbox(
    value=False,
    description='Enable TG Notifications',
    indent=False,
    layout=widgets.Layout(width='200px')
)

tg_token_input = widgets.Password(
    placeholder='Telegram Bot Token',
    description='Token:',
    style={'description_width': '60px'},
    layout=widgets.Layout(width='300px')
)

tg_chat_id_input = widgets.Text(
    placeholder='Telegram Chat ID',
    description='Chat ID:',
    style={'description_width': '60px'},
    layout=widgets.Layout(width='200px')
)

settings_box = widgets.HBox([
    widgets.VBox([
        widgets.HTML('<b style="font-size:14px;">4️⃣ Parameters</b>'),
        widgets.HBox([profit_threshold, check_interval], layout=widgets.Layout(gap='30px'))
    ]),
    widgets.VBox([
        widgets.HTML('<b style="font-size:14px;">5️⃣ Telegram</b>'),
        widgets.HBox([enable_telegram], layout=widgets.Layout(gap='10px')),
        widgets.HBox([tg_token_input, tg_chat_id_input], layout=widgets.Layout(gap='10px'))
    ])
], layout=widgets.Layout(gap='60px', margin='10px 0'))


## ▶️ Run Button
run_button = widgets.Button(
    description='🚀 Start Detection',
    button_style='success',
    layout=widgets.Layout(width='180px', height='40px')
)

button_box = widgets.HBox([run_button], layout=widgets.Layout(justify_content='center', margin='15px 0'))


## 🖥️ Output Areas
table_output = widgets.Output(layout=widgets.Layout(width='100%', border='2px solid #3CB371', padding='15px', height='500px', overflow_y='auto'))
output_area = widgets.Output(layout=widgets.Layout(width='100%', border='2px solid #999', padding='15px', height='1000px', overflow_y='auto'))


# --- Main Layout ---
main_layout = widgets.VBox([
    title,
    widgets.HTML('<hr style="margin: 10px 0; border: 1px solid #ddd;">'),
    row1,
    widgets.HTML('<hr style="margin: 10px 0; border: 1px solid #ddd;">'),
    fee_box,
    widgets.HTML('<hr style="margin: 10px 0; border: 1px solid #ddd;">'),
    settings_box,
    widgets.HTML('<hr style="margin: 10px 0; border: 1px solid #ddd;">'),
    button_box,
    widgets.HTML('<b>📈 Arbitrage Opportunities (Top 10)</b>'),
    table_output,
    widgets.HTML('<b>💻 Log / Console Output</b>'),
    output_area,
], layout=widgets.Layout(width='100%'))


# --- Event Handler Logic ---

# 修正後的 TG 函式：使用 RequestException 和非阻塞設計
def send_telegram_notification(token, chat_id, msg):
    """將訊息發送到 Telegram，使用 HTML 格式。"""
    if not token or not chat_id: return
    try:
        url = f'https://api.telegram.org/bot{token}/sendMessage'
        payload = {'chat_id': chat_id, 'text': msg, 'parse_mode': 'HTML'}
        # 在獨立的 TG 線程池中進行同步 POST 請求
        response = requests.post(url, json=payload, timeout=5)
        response.raise_for_status() # 檢查 HTTP 狀態碼 (200-399)

    except RequestException as e:
        # 捕捉與請求相關的所有錯誤 (連接、超時、HTTP 錯誤等)
        error_msg = f'❌ TG Network/HTTP Error: {type(e).__name__} - {str(e)[:50]}...'
        with output_area: print(error_msg)
    except Exception as e:
        # 捕捉其他意外錯誤
        error_msg = f'❌ TG General Error: {type(e).__name__} - {str(e)[:50]}...'
        with output_area: print(error_msg)


def get_current_fees():
    return {ex: fee_widgets[ex].value for ex in ALL_EXCHANGES}

# 新增: 運行狀態鎖定，確保多執行緒安全
running = False
running_lock = threading.Lock()

def on_run_clicked(b):
    global running, detector

    # 修正: 使用鎖確保 running 旗標在多執行緒中安全更新
    with running_lock:
        if running:
            with output_area: clear_output(); print('⚠️ Detection is already running. Please interrupt the kernel to stop.')
            return
        running = True

    run_button.disabled = True

    # 獲取配置參數
    exchanges_str = exchange_display.value
    cryptos_symbols_str = crypto_display.value

    exchanges = [e.strip() for e in exchanges_str.split(',') if e.strip() and e.strip() != '(None)']
    selected_symbols = [s.strip() for s in cryptos_symbols_str.split('\n') if s.strip() and s.strip() != '(None)']
    selected_cryptos = [c for c in CRYPTO_PAIRS if c['symbol'] in selected_symbols]


    threshold = profit_threshold.value
    interval = check_interval.value
    enable_tg = enable_telegram.value
    tg_token = tg_token_input.value if enable_tg else ''
    tg_chat_id = tg_chat_id_input.value if enable_tg else ''

    # 檢查配置
    if not exchanges:
        with output_area: clear_output(); print('❌ Select exchanges first')
        with running_lock: running = False; run_button.disabled = False; return
    if not selected_cryptos:
        with output_area: clear_output(); print('❌ Select cryptos first')
        with running_lock: running = False; run_button.disabled = False; return
    if enable_tg and (not tg_token or not tg_chat_id):
        with output_area: clear_output(); print('❌ TG config incomplete (Token or Chat ID missing)')
        with running_lock: running = False; run_button.disabled = False; return

    # 初始化 CCXT 客戶端和費用
    detector.fees = get_current_fees()
    detector.initialize_exchanges(exchanges) # 初始化 CCXT 客戶端

    with output_area:
        clear_output()
        print('--- Arbitrage Detector Initialized ---')
        print(f'🚨 Stop: interrupt Jupyter kernel')
        print(f'Exchanges: {len(exchanges)} | Cryptos: {len(selected_cryptos)}')
        print(f'Min Profit: {threshold}% | Interval: {interval}s')
        print(f'Telegram: {"Enabled" if enable_tg else "Disabled"}')
        print('------------------------------------')


    iteration = 0
    while True: # 修正: 外部的 while 迴圈必須檢查 running 狀態
        with running_lock:
            if not running: break

        iteration += 1
        round_opportunities = []

        with output_area:
            clear_output(wait=True)
            print(f'🔄 #{iteration} | {datetime.now().strftime("%Y-%m-%d %H:%M:%S")} | Fetching Prices via CCXT...')

        # 使用線程池並發獲取價格
        # 最大線程數調整為交易所*幣種數，避免過度線程化
        max_price_workers = len(exchanges) * len(selected_cryptos)
        with ThreadPoolExecutor(max_workers=max_price_workers) as executor:
            # 傳遞完整的 crypto 物件和選中的交易所列表
            futures = {executor.submit(detector.get_crypto_all_prices, crypto, exchanges): crypto for crypto in selected_cryptos}

            for future in as_completed(futures):
                with running_lock:
                    if not running: break

                crypto = futures[future]
                unified_symbol = crypto['symbol']

                try:
                    price_data = future.result()

                    # 檢查是否收到有效數據
                    if not price_data or not price_data.get('prices') or len(price_data['prices']) < 2:
                        with output_area: print(f'⚠️ {unified_symbol}: Not enough valid price data received (skipping calculation)')
                        continue

                    # Log 顯示 CCXT 抓到的數據價格
                    log_msg = f"🔎 {unified_symbol} Prices:"
                    for ex, p in price_data['prices'].items():
                        ask = f"{p['ask']:.6f}" if p['ask'] is not None else 'N/A'
                        bid = f"{p['bid']:.6f}" if p['bid'] is not None else 'N/A'
                        log_msg += f" {ex[:4]}={ask}/{bid}"
                    with output_area: print(log_msg)

                    all_pairs = detector.find_all_arbitrage_pairs(price_data)
                    profitable_pairs = [p for p in all_pairs if p['profit_percent'] >= threshold]

                    if profitable_pairs:
                        profitable_pairs.sort(key=lambda x: x['profit_percent'], reverse=True)
                        round_opportunities.extend(profitable_pairs)
                        with output_area: print(f'✅ {unified_symbol}: Found {len(profitable_pairs)} opportunities, Max: {profitable_pairs[0]["profit_percent"]:.4f}%')

                        if enable_tg:
                            best = profitable_pairs[0]
                            # 修正: 將 TG 通知提交給獨立的線程池，使其非阻塞
                            msg = f'🔔 <b>套利機會: {crypto["name"]} ({best["profit_percent"]:.4f}%)</b>\n買入: {best["buy_exchange"].capitalize()} @ ${best["buy_price"]:.6f}\n賣出: {best["sell_exchange"].capitalize()} @ ${best["sell_price"]:.6f}'
                            detector.tg_executor.submit(send_telegram_notification, tg_token, tg_chat_id, msg)
                    else:
                        with output_area: print(f'ℹ️ {unified_symbol}: No opportunities > {threshold}%')

                except Exception as e:
                    # 捕捉線程執行期間的任何意外錯誤
                    with output_area: print(f'❌ {unified_symbol} General Error during processing: {str(e)}')

            with running_lock:
                if not running: break

        if not running: break

        # 更新套利機會表格 (與原程式碼相同)
        with table_output:
            clear_output(wait=True)
            if round_opportunities:
                # 僅顯示前 10 筆
                df = pd.DataFrame(round_opportunities).sort_values('profit_percent', ascending=False).head(10)
                df_display = df[['symbol', 'buy_exchange', 'sell_exchange', 'buy_price', 'sell_price', 'buy_fee', 'sell_fee', 'profit_percent']].copy()

                df_display.columns = ['Crypto', 'Buy', 'Sell', 'Ask Price (Buy)', 'Bid Price (Sell)', 'Buy Fee (%)', 'Sell Fee (%)', 'Profit (%)']

                # 格式化數值
                df_display['Profit (%)'] = df_display['Profit (%)'].round(4)
                df_display['Buy Fee (%)'] = (df_display['Buy Fee (%)'] * 100).round(4)
                df_display['Sell Fee (%)'] = (df_display['Sell Fee (%)'] * 100).round(4)
                df_display['Ask Price (Buy)'] = df_display['Ask Price (Buy)'].round(6)
                df_display['Bid Price (Sell)'] = df_display['Bid Price (Sell)'].round(6)

                html_table = df_display.to_html(index=False, classes='table table-striped', float_format='%.6f')
                display(HTML(f'<b>Found Total: {len(round_opportunities)} | Displaying Top {len(df_display)}</b><br>{html_table}'))
            else:
                # 最終總結：如果 round_opportunities 為空，則輸出此訊息
                print('ℹ️ No opportunities found above the minimum profit threshold.')

        # 延遲
        time.sleep(interval)

    # 迴圈結束後的處理
    with output_area: print('⏹ Stopped.')
    run_button.disabled = False


run_button.on_click(on_run_clicked)
display(main_layout)

In [1]:
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import pandas as pd
from datetime import datetime
import time
import requests
from concurrent.futures import ThreadPoolExecutor, as_completed
import ccxt
import sys
import matplotlib.pyplot as plt
import random
import threading
from requests.exceptions import RequestException

# ----------------------------------------------------
# 📌 終端字體設定 (根據您的作業系統需求)
# 設定中文字體
plt.rcParams['font.sans-serif'] = ['WenQuanYi Micro Hei']
plt.rcParams['axes.unicode_minus'] = False
# ----------------------------------------------------

# --- Config and Utilities ---
CRYPTO_PAIRS = [
    {'symbol': 'BTC/USDT', 'name': 'Bitcoin'},
    {'symbol': 'ETH/USDT', 'name': 'Ethereum'},
    {'symbol': 'XRP/USDT', 'name': 'Ripple'},
    {'symbol': 'DOGE/USDT', 'name': 'Dogecoin'},
    {'symbol': 'SOL/USDT', 'name': 'Solana'},
    {'symbol': 'ADA/USDT', 'name': 'Cardano'},
    {'symbol': 'DOT/USDT', 'name': 'Polkadot'},
    {'symbol': 'LTC/USDT', 'name': 'Litecoin'},
    {'symbol': 'BCH/USDT', 'name': 'Bitcoin Cash'},
    {'symbol': 'LINK/USDT', 'name': 'Chainlink'},
    {'symbol': 'VET/USDT', 'name': 'VeChain'},
    {'symbol': 'TRX/USDT', 'name': 'TRON'},
    {'symbol': 'MATIC/USDT', 'name': 'Polygon'},
    {'symbol': 'AVAX/USDT', 'name': 'Avalanche'},
]

ALL_EXCHANGES = ['binance', 'bybit', 'okx', 'kucoin', 'huobi', 'gate', 'kraken', 'coinbase', 'bitfinex']

class ArbitrageDetector:
    def __init__(self):
        self.fees = {}
        self.clients = {}
        self.unified_symbols = {c['symbol']: c['name'] for c in CRYPTO_PAIRS}

    def initialize_exchanges(self, exchanges):
        """初始化 CCXT 交易所客戶端並啟用速率限制。"""
        self.clients = {}
        timeout_ms = 10000
        for exchange_id in exchanges:
            try:
                exchange_class = getattr(ccxt, exchange_id)
                self.clients[exchange_id] = exchange_class({
                    'timeout': timeout_ms,
                    'enableRateLimit': True,
                })
            except Exception as e:
                pass

    def get_crypto_all_prices(self, crypto, exchanges):
        """使用 CCXT 獲取指定交易對在各交易所的真實買一價和賣一價。"""
        prices = {}
        unified_symbol = crypto['symbol']

        for exchange_id in exchanges:
            exchange = self.clients.get(exchange_id)
            if not exchange:
                continue

            try:
                ticker = exchange.fetch_ticker(unified_symbol)

                bid_price = ticker.get('bid')
                ask_price = ticker.get('ask')

                if bid_price is not None and ask_price is not None:
                    prices[exchange_id] = {
                        'bid': bid_price,
                        'ask': ask_price,
                    }

            except ccxt.ExchangeNotAvailable:
                pass
            except ccxt.DDoSProtection or ccxt.RateLimitExceeded:
                pass
            except Exception:
                pass

        return {
            'symbol': unified_symbol,
            'name': crypto['name'],
            'prices': prices
        }

    def find_all_arbitrage_pairs(self, price_data):
        """計算所有套利機會。"""
        opportunities = []
        symbol = price_data['symbol']
        exchanges = list(price_data['prices'].keys())

        current_fees = get_current_fees()

        for i in range(len(exchanges)):
            for j in range(len(exchanges)):
                if i == j:
                    continue

                buy_ex = exchanges[i]
                sell_ex = exchanges[j]

                buy_price = price_data['prices'][buy_ex].get('ask')
                sell_price = price_data['prices'][sell_ex].get('bid')

                buy_fee = current_fees.get(buy_ex, 0)
                sell_fee = current_fees.get(sell_ex, 0)

                if buy_price and sell_price and buy_price > 0:
                    
                    net_sell_price = sell_price * (1 - sell_fee)
                    net_profit_ratio = (net_sell_price - buy_price) / buy_price
                    profit_percent = net_profit_ratio * 100

                    if profit_percent >= profit_threshold.value:
                        opportunities.append({
                            'symbol': symbol,
                            'buy_exchange': buy_ex,
                            'sell_exchange': sell_ex,
                            'buy_price': buy_price,
                            'sell_price': sell_price,
                            'buy_fee': buy_fee,
                            'sell_fee': sell_fee,
                            'net_sell_price': net_sell_price,
                            'profit_percent': profit_percent,
                        })
        return opportunities

detector = ArbitrageDetector()

# --- UI Component Definition ---

## 🏆 Title
title = widgets.HTML(
    '<div style="background-color:#6A5ACD; color:white; padding:10px; border-radius:8px; text-align:center;">'
    '<h2>🚀 Cross-Exchange Arbitrage Detector (CCXT)</h2>'
    '</div>'
)

## 1️⃣ Exchange Selection
exchange_selector = widgets.SelectMultiple(
    options=ALL_EXCHANGES,
    value=['binance', 'bybit', 'okx'],
    description='',
    layout=widgets.Layout(width='150px', height='250px')
)

exchange_display = widgets.Textarea(
    value='binance, bybit, okx',
    placeholder='Selected...',
    disabled=True,
    layout=widgets.Layout(width='180px', height='250px')
)

exchange_button = widgets.Button(
    description='✅ Confirm',
    button_style='success',
    layout=widgets.Layout(width='150px', height='30px')
)

def update_exchange_display(b):
    exchange_display.value = ', '.join(exchange_selector.value) if exchange_selector.value else '(None)'
exchange_button.on_click(update_exchange_display)

exchange_box = widgets.VBox([
    widgets.HTML('<b style="font-size:14px;">1️⃣ Exchanges</b>'),
    widgets.HBox([
        widgets.VBox([widgets.HTML('<span style="font-size:12px;">Available:</span>'), exchange_selector]),
        widgets.VBox([widgets.HTML('<span style="font-size:12px;">Selected:</span>'), exchange_display]),
    ], layout=widgets.Layout(gap='10px')),
    exchange_button
])


## 2️⃣ Crypto Selection
crypto_symbols = [f"{c['symbol']} - {c['name']}" for c in CRYPTO_PAIRS]
crypto_selector = widgets.SelectMultiple(
    options=crypto_symbols,
    value=[crypto_symbols[0], crypto_symbols[1], crypto_symbols[2]] if len(crypto_symbols) >= 3 else crypto_symbols,
    description='',
    layout=widgets.Layout(width='200px', height='250px')
)

selected_initial_symbols = [text.split(' - ')[0] for text in crypto_selector.value]
crypto_display = widgets.Textarea(
    value='\n'.join(selected_initial_symbols),
    placeholder='Selected...',
    disabled=True,
    layout=widgets.Layout(width='120px', height='250px')
)

crypto_button = widgets.Button(
    description='✅ Confirm',
    button_style='success',
    layout=widgets.Layout(width='150px', height='30px')
)

def update_crypto_display(b):
    selected_symbols = [text.split(' - ')[0] for text in crypto_selector.value]
    crypto_display.value = '\n'.join(selected_symbols) if selected_symbols else '(None)'
crypto_button.on_click(update_crypto_display)

crypto_box = widgets.VBox([
    widgets.HTML('<b style="font-size:14px;">2️⃣ Cryptos</b>'),
    widgets.HBox([
        widgets.VBox([widgets.HTML('<span style="font-size:12px;">Available:</span>'), crypto_selector]),
        widgets.VBox([widgets.HTML('<span style="font-size:12px;">Selected:</span>'), crypto_display]),
    ], layout=widgets.Layout(gap='10px')),
    crypto_button
])

row1 = widgets.HBox([exchange_box, crypto_box], layout=widgets.Layout(gap='50px', margin='5px 0'))


## 🔧 Taker Fees
fee_widgets = {}
initial_fees = {ex: 0.001 for ex in ALL_EXCHANGES}
initial_fees['bybit'] = 0.0007
initial_fees['okx'] = 0.0008

for ex in ALL_EXCHANGES:
    fee_widgets[ex] = widgets.BoundedFloatText(
        value=initial_fees.get(ex, 0.001),
        min=0.0,
        max=0.01,
        step=0.0001,
        description=ex.capitalize(),
        style={'description_width': '80px'},
        layout=widgets.Layout(width='160px')
    )

fee_row1 = widgets.HBox([fee_widgets[ex] for ex in ALL_EXCHANGES[:5]], layout=widgets.Layout(gap='15px'))
fee_row2 = widgets.HBox([fee_widgets[ex] for ex in ALL_EXCHANGES[5:]], layout=widgets.Layout(gap='15px'))

fee_box = widgets.VBox([
    widgets.HTML('<b style="font-size:14px;">3️⃣ Taker Fees:</b>'),
    fee_row1,
    fee_row2
], layout=widgets.Layout(border='1px solid #555', padding='10px', margin='10px 0'))


## 4️⃣ Parameters & Telegram
profit_threshold = widgets.FloatSlider(
    value=0.5,
    min=0.01,
    max=5.0,
    step=0.01,
    description='Min Profit %',
    style={'description_width': '100px'},
    readout_format='.2f',
    layout=widgets.Layout(width='350px')
)

check_interval = widgets.Dropdown(
    options={'30s': 30, '1min': 60, '5min': 300, '10min': 600, '30min': 1800, '1hr': 3600},
    value=30,
    description='Interval',
    style={'description_width': '100px'},
    layout=widgets.Layout(width='220px')
)

enable_telegram = widgets.Checkbox(
    value=False,
    description='Enable TG Notifications',
    indent=False,
    layout=widgets.Layout(width='200px')
)

tg_token_input = widgets.Password(
    placeholder='Telegram Bot Token',
    description='Token:',
    style={'description_width': '60px'},
    layout=widgets.Layout(width='300px')
)

tg_chat_id_input = widgets.Text(
    placeholder='Telegram Chat ID',
    description='Chat ID:',
    style={'description_width': '60px'},
    layout=widgets.Layout(width='200px')
)

settings_box = widgets.HBox([
    widgets.VBox([
        widgets.HTML('<b style="font-size:14px;">4️⃣ Parameters</b>'),
        widgets.HBox([profit_threshold, check_interval], layout=widgets.Layout(gap='30px'))
    ]),
    widgets.VBox([
        widgets.HTML('<b style="font-size:14px;">5️⃣ Telegram</b>'),
        widgets.HBox([enable_telegram], layout=widgets.Layout(gap='10px')),
        widgets.HBox([tg_token_input, tg_chat_id_input], layout=widgets.Layout(gap='10px'))
    ])
], layout=widgets.Layout(gap='60px', margin='10px 0'))


## ▶️ Run Button
run_button = widgets.Button(
    description='🚀 Start Detection',
    button_style='success',
    layout=widgets.Layout(width='180px', height='40px')
)

button_box = widgets.HBox([run_button], layout=widgets.Layout(justify_content='center', margin='15px 0'))


## 🖥️ Output Areas
table_output = widgets.Output(layout=widgets.Layout(width='100%', border='2px solid #3CB371', padding='15px', height='500px', overflow_y='auto'))
output_area = widgets.Output(layout=widgets.Layout(width='100%', border='2px solid #999', padding='15px', height='1000px', overflow_y='auto'))


# --- Main Layout ---
main_layout = widgets.VBox([
    title,
    widgets.HTML('<hr style="margin: 10px 0; border: 1px solid #ddd;">'),
    row1,
    widgets.HTML('<hr style="margin: 10px 0; border: 1px solid #ddd;">'),
    fee_box,
    widgets.HTML('<hr style="margin: 10px 0; border: 1px solid #ddd;">'),
    settings_box,
    widgets.HTML('<hr style="margin: 10px 0; border: 1px solid #ddd;">'),
    button_box,
    widgets.HTML('<b>📈 Arbitrage Opportunities (Top 10)</b>'),
    table_output,
    widgets.HTML('<b>💻 Log / Console Output</b>'),
    output_area,
], layout=widgets.Layout(width='100%'))


# --- Event Handler Logic ---

def send_telegram_notification(token, chat_id, msg):
    """將訊息發送到 Telegram，使用 HTML 格式。"""
    if not token or not chat_id:
        print('❌ TG Error: Token or Chat ID is empty')
        return
    try:
        url = f'https://api.telegram.org/bot{token}/sendMessage'
        payload = {'chat_id': chat_id, 'text': msg, 'parse_mode': 'HTML'}
        response = requests.post(url, json=payload, timeout=10)
        
        if response.status_code == 200:
            print(f'✅ TG 發送成功')
        else:
            print(f'❌ TG API: {response.status_code} - {response.text}')

    except Exception as e:
        print(f'❌ TG Error: {e}')


def get_current_fees():
    return {ex: fee_widgets[ex].value for ex in ALL_EXCHANGES}

running = False
running_lock = threading.Lock()

def on_run_clicked(b):
    global running, detector

    with running_lock:
        if running:
            with output_area: clear_output(); print('⚠️ Detection is already running. Please interrupt the kernel to stop.')
            return
        running = True

    run_button.disabled = True

    exchanges_str = exchange_display.value
    cryptos_symbols_str = crypto_display.value

    exchanges = [e.strip() for e in exchanges_str.split(',') if e.strip() and e.strip() != '(None)']
    selected_symbols = [s.strip() for s in cryptos_symbols_str.split('\n') if s.strip() and s.strip() != '(None)']
    selected_cryptos = [c for c in CRYPTO_PAIRS if c['symbol'] in selected_symbols]


    threshold = profit_threshold.value
    interval = check_interval.value
    enable_tg = enable_telegram.value
    tg_token = tg_token_input.value if enable_tg else ''
    tg_chat_id = tg_chat_id_input.value if enable_tg else ''

    if not exchanges:
        with output_area: clear_output(); print('❌ Select exchanges first')
        with running_lock: running = False; run_button.disabled = False; return
    if not selected_cryptos:
        with output_area: clear_output(); print('❌ Select cryptos first')
        with running_lock: running = False; run_button.disabled = False; return
    if enable_tg and (not tg_token or not tg_chat_id):
        with output_area: clear_output(); print('❌ TG config incomplete (Token or Chat ID missing)')
        with running_lock: running = False; run_button.disabled = False; return

    detector.fees = get_current_fees()
    detector.initialize_exchanges(exchanges)

    with output_area:
        clear_output()
        print('--- Arbitrage Detector Initialized ---')
        print(f'🚨 Stop: interrupt Jupyter kernel')
        print(f'Exchanges: {len(exchanges)} | Cryptos: {len(selected_cryptos)}')
        print(f'Min Profit: {threshold}% | Interval: {interval}s')
        print(f'Telegram: {"Enabled" if enable_tg else "Disabled"}')
        print('------------------------------------')


    iteration = 0
    while True:
        with running_lock:
            if not running: break

        iteration += 1
        round_opportunities = []

        with output_area:
            clear_output(wait=True)
            print(f'🔄 #{iteration} | {datetime.now().strftime("%Y-%m-%d %H:%M:%S")} | Fetching Prices via CCXT...')

        max_price_workers = len(exchanges) * len(selected_cryptos)
        with ThreadPoolExecutor(max_workers=max_price_workers) as executor:
            futures = {executor.submit(detector.get_crypto_all_prices, crypto, exchanges): crypto for crypto in selected_cryptos}

            for future in as_completed(futures):
                with running_lock:
                    if not running: break

                crypto = futures[future]
                unified_symbol = crypto['symbol']

                try:
                    price_data = future.result()

                    if not price_data or not price_data.get('prices') or len(price_data['prices']) < 2:
                        with output_area: print(f'⚠️ {unified_symbol}: Not enough valid price data received (skipping calculation)')
                        continue

                    log_msg = f"🔎 {unified_symbol} Prices:"
                    for ex, p in price_data['prices'].items():
                        ask = f"{p['ask']:.6f}" if p['ask'] is not None else 'N/A'
                        bid = f"{p['bid']:.6f}" if p['bid'] is not None else 'N/A'
                        log_msg += f" {ex[:4]}={ask}/{bid}"
                    with output_area: print(log_msg)

                    all_pairs = detector.find_all_arbitrage_pairs(price_data)
                    profitable_pairs = [p for p in all_pairs if p['profit_percent'] >= threshold]

                    if profitable_pairs:
                        profitable_pairs.sort(key=lambda x: x['profit_percent'], reverse=True)
                        round_opportunities.extend(profitable_pairs)
                        with output_area: print(f'✅ {unified_symbol}: Found {len(profitable_pairs)} opportunities, Max: {profitable_pairs[0]["profit_percent"]:.4f}%')

                        if enable_tg:
                            best = profitable_pairs[0]
                            msg = f'🔔 <b>套利機會: {crypto["name"]} ({best["profit_percent"]:.4f}%)</b>\n買入: {best["buy_exchange"].capitalize()} @ ${best["buy_price"]:.6f}\n賣出: {best["sell_exchange"].capitalize()} @ ${best["sell_price"]:.6f}'
                            # 直接呼叫，不用線程池
                            send_telegram_notification(tg_token, tg_chat_id, msg)
                    else:
                        with output_area: print(f'ℹ️ {unified_symbol}: No opportunities > {threshold}%')

                except Exception as e:
                    with output_area: print(f'❌ {unified_symbol} General Error during processing: {str(e)}')

            with running_lock:
                if not running: break

        if not running: break

        with table_output:
            clear_output(wait=True)
            if round_opportunities:
                df = pd.DataFrame(round_opportunities).sort_values('profit_percent', ascending=False).head(10)
                df_display = df[['symbol', 'buy_exchange', 'sell_exchange', 'buy_price', 'sell_price', 'buy_fee', 'sell_fee', 'profit_percent']].copy()

                df_display.columns = ['Crypto', 'Buy', 'Sell', 'Ask Price (Buy)', 'Bid Price (Sell)', 'Buy Fee (%)', 'Sell Fee (%)', 'Profit (%)']

                df_display['Profit (%)'] = df_display['Profit (%)'].round(4)
                df_display['Buy Fee (%)'] = (df_display['Buy Fee (%)'] * 100).round(4)
                df_display['Sell Fee (%)'] = (df_display['Sell Fee (%)'] * 100).round(4)
                df_display['Ask Price (Buy)'] = df_display['Ask Price (Buy)'].round(6)
                df_display['Bid Price (Sell)'] = df_display['Bid Price (Sell)'].round(6)

                html_table = df_display.to_html(index=False, classes='table table-striped', float_format='%.6f')
                display(HTML(f'<b>Found Total: {len(round_opportunities)} | Displaying Top {len(df_display)}</b><br>{html_table}'))
            else:
                print('ℹ️ No opportunities found above the minimum profit threshold.')

        time.sleep(interval)

    with output_area: print('⏹ Stopped.')
    run_button.disabled = False


run_button.on_click(on_run_clicked)
display(main_layout)

In [2]:
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import pandas as pd
from datetime import datetime
import time
import requests
from concurrent.futures import ThreadPoolExecutor, as_completed
import ccxt
import sys
import matplotlib.pyplot as plt
import random
import threading
from requests.exceptions import RequestException

# ----------------------------------------------------
# 📌 Terminal Font Configuration (Based on your Ubuntu setup for Chinese display)
# Setting Chinese font for matplotlib
try:
    plt.rcParams['font.sans-serif'] = ['WenQuanYi Micro Hei']
    plt.rcParams['axes.unicode_minus'] = False
except Exception:
    # Fallback in case font is not found
    pass
# ----------------------------------------------------

# --- Config and Utilities ---
CRYPTO_PAIRS = [
    {'symbol': 'BTC/USDT', 'name': 'Bitcoin'},
    {'symbol': 'ETH/USDT', 'name': 'Ethereum'},
    {'symbol': 'XRP/USDT', 'name': 'Ripple'},
    {'symbol': 'DOGE/USDT', 'name': 'Dogecoin'},
    {'symbol': 'SOL/USDT', 'name': 'Solana'},
    {'symbol': 'ADA/USDT', 'name': 'Cardano'},
    {'symbol': 'DOT/USDT', 'name': 'Polkadot'},
    {'symbol': 'LTC/USDT', 'name': 'Litecoin'},
    {'symbol': 'BCH/USDT', 'name': 'Bitcoin Cash'},
    {'symbol': 'LINK/USDT', 'name': 'Chainlink'},
    {'symbol': 'VET/USDT', 'name': 'VeChain'},
    {'symbol': 'TRX/USDT', 'name': 'TRON'},
    {'symbol': 'MATIC/USDT', 'name': 'Polygon'},
    {'symbol': 'AVAX/USDT', 'name': 'Avalanche'},
]

ALL_EXCHANGES = ['binance', 'bybit', 'okx', 'kucoin', 'huobi', 'gate', 'kraken', 'coinbase', 'bitfinex']

class ArbitrageDetector:
    def __init__(self):
        self.fees = {}
        self.clients = {}
        self.unified_symbols = {c['symbol']: c['name'] for c in CRYPTO_PAIRS}

    def initialize_exchanges(self, exchanges):
        """Initializes CCXT exchange clients and enables rate limiting."""
        self.clients = {}
        timeout_ms = 10000
        for exchange_id in exchanges:
            try:
                exchange_class = getattr(ccxt, exchange_id)
                self.clients[exchange_id] = exchange_class({
                    'timeout': timeout_ms,
                    'enableRateLimit': True,
                })
            except Exception as e:
                pass

    def get_crypto_all_prices(self, crypto, exchanges):
        """Fetches the real bid and ask prices from exchanges using CCXT."""
        prices = {}
        unified_symbol = crypto['symbol']

        for exchange_id in exchanges:
            exchange = self.clients.get(exchange_id)
            if not exchange:
                continue

            try:
                # Use a shorter timeout for individual fetches within the thread
                ticker = exchange.fetch_ticker(unified_symbol, params={'timeout': 5000})

                bid_price = ticker.get('bid')
                ask_price = ticker.get('ask')

                if bid_price is not None and ask_price is not None:
                    prices[exchange_id] = {
                        'bid': bid_price,
                        'ask': ask_price,
                    }

            except ccxt.ExchangeNotAvailable:
                pass
            except ccxt.DDoSProtection or ccxt.RateLimitExceeded:
                pass
            except Exception:
                pass

        return {
            'symbol': unified_symbol,
            'name': crypto['name'],
            'prices': prices
        }

    def find_all_arbitrage_pairs(self, price_data):
        """Calculates all arbitrage opportunities."""
        opportunities = []
        symbol = price_data['symbol']
        exchanges = list(price_data['prices'].keys())

        current_fees = get_current_fees()

        for i in range(len(exchanges)):
            for j in range(len(exchanges)):
                if i == j:
                    continue

                buy_ex = exchanges[i]
                sell_ex = exchanges[j]

                buy_price = price_data['prices'][buy_ex].get('ask')
                sell_price = price_data['prices'][sell_ex].get('bid')

                # Taker Fee
                buy_fee = current_fees.get(buy_ex, 0)
                sell_fee = current_fees.get(sell_ex, 0)

                if buy_price and sell_price and buy_price > 0:
                    
                    # Net price = Sell Price * (1 - Taker Fee)
                    net_sell_price = sell_price * (1 - sell_fee)
                    
                    # Net Profit Ratio (considering selling fee)
                    net_profit_ratio = (net_sell_price - buy_price) / buy_price
                    profit_percent = net_profit_ratio * 100

                    if profit_percent >= profit_threshold.value:
                        opportunities.append({
                            'symbol': symbol,
                            'name': price_data['name'], 
                            'buy_exchange': buy_ex,
                            'sell_exchange': sell_ex,
                            'buy_price': buy_price,
                            'sell_price': sell_price,
                            'buy_fee': buy_fee,
                            'sell_fee': sell_fee,
                            'net_sell_price': net_sell_price,
                            'profit_percent': profit_percent,
                        })
        return opportunities

detector = ArbitrageDetector()

# --- UI Component Definition (Unchanged) ---

## 🏆 Title
title = widgets.HTML(
    '<div style="background-color:#6A5ACD; color:white; padding:10px; border-radius:8px; text-align:center;">'
    '<h2>🚀 Cross-Exchange Arbitrage Detector (CCXT)</h2>'
    '</div>'
)

## 1️⃣ Exchange Selection
exchange_selector = widgets.SelectMultiple(
    options=ALL_EXCHANGES,
    value=['binance', 'bybit', 'okx'],
    description='',
    layout=widgets.Layout(width='150px', height='250px')
)

exchange_display = widgets.Textarea(
    value='binance, bybit, okx',
    placeholder='Selected...',
    disabled=True,
    layout=widgets.Layout(width='180px', height='250px')
)

exchange_button = widgets.Button(
    description='✅ Confirm',
    button_style='success',
    layout=widgets.Layout(width='150px', height='30px')
)

def update_exchange_display(b):
    exchange_display.value = ', '.join(exchange_selector.value) if exchange_selector.value else '(None)'
exchange_button.on_click(update_exchange_display)

exchange_box = widgets.VBox([
    widgets.HTML('<b style="font-size:14px;">1️⃣ Exchanges</b>'),
    widgets.HBox([
        widgets.VBox([widgets.HTML('<span style="font-size:12px;">Available:</span>'), exchange_selector]),
        widgets.VBox([widgets.HTML('<span style="font-size:12px;">Selected:</span>'), exchange_display]),
    ], layout=widgets.Layout(gap='10px')),
    exchange_button
])


## 2️⃣ Crypto Selection
crypto_symbols = [f"{c['symbol']} - {c['name']}" for c in CRYPTO_PAIRS]
crypto_selector = widgets.SelectMultiple(
    options=crypto_symbols,
    value=[crypto_symbols[0], crypto_symbols[1], crypto_symbols[2]] if len(crypto_symbols) >= 3 else crypto_symbols,
    description='',
    layout=widgets.Layout(width='200px', height='250px')
)

selected_initial_symbols = [text.split(' - ')[0] for text in crypto_selector.value]
crypto_display = widgets.Textarea(
    value='\n'.join(selected_initial_symbols),
    placeholder='Selected...',
    disabled=True,
    layout=widgets.Layout(width='120px', height='250px')
)

crypto_button = widgets.Button(
    description='✅ Confirm',
    button_style='success',
    layout=widgets.Layout(width='150px', height='30px')
)

def update_crypto_display(b):
    selected_symbols = [text.split(' - ')[0] for text in crypto_selector.value]
    crypto_display.value = '\n'.join(selected_symbols) if selected_symbols else '(None)'
crypto_button.on_click(update_crypto_display)

crypto_box = widgets.VBox([
    widgets.HTML('<b style="font-size:14px;">2️⃣ Cryptos</b>'),
    widgets.HBox([
        widgets.VBox([widgets.HTML('<span style="font-size:12px;">Available:</span>'), crypto_selector]),
        widgets.VBox([widgets.HTML('<span style="font-size:12px;">Selected:</span>'), crypto_display]),
    ], layout=widgets.Layout(gap='10px')),
    crypto_button
])

row1 = widgets.HBox([exchange_box, crypto_box], layout=widgets.Layout(gap='50px', margin='5px 0'))


## 🔧 Taker Fees
fee_widgets = {}
initial_fees = {ex: 0.001 for ex in ALL_EXCHANGES}
initial_fees['bybit'] = 0.0007
initial_fees['okx'] = 0.0008

for ex in ALL_EXCHANGES:
    fee_widgets[ex] = widgets.BoundedFloatText(
        value=initial_fees.get(ex, 0.001),
        min=0.0,
        max=0.01,
        step=0.0001,
        description=ex.capitalize(),
        style={'description_width': '80px'},
        layout=widgets.Layout(width='160px')
    )

fee_row1 = widgets.HBox([fee_widgets[ex] for ex in ALL_EXCHANGES[:5]], layout=widgets.Layout(gap='15px'))
fee_row2 = widgets.HBox([fee_widgets[ex] for ex in ALL_EXCHANGES[5:]], layout=widgets.Layout(gap='15px'))

fee_box = widgets.VBox([
    widgets.HTML('<b style="font-size:14px;">3️⃣ Taker Fees:</b>'),
    fee_row1,
    fee_row2
], layout=widgets.Layout(border='1px solid #555', padding='10px', margin='10px 0'))


## 4️⃣ Parameters & Telegram
profit_threshold = widgets.FloatSlider(
    value=0.5,
    min=0.01,
    max=5.0,
    step=0.01,
    description='Min Profit %',
    style={'description_width': '100px'},
    readout_format='.2f',
    layout=widgets.Layout(width='350px')
)

check_interval = widgets.Dropdown(
    options={'30s': 30, '1min': 60, '5min': 300, '10min': 600, '30min': 1800, '1hr': 3600},
    value=30,
    description='Interval',
    style={'description_width': '100px'},
    layout=widgets.Layout(width='220px')
)

enable_telegram = widgets.Checkbox(
    value=False,
    description='Enable TG Notifications',
    indent=False,
    layout=widgets.Layout(width='200px')
)

tg_token_input = widgets.Password(
    placeholder='Telegram Bot Token',
    description='Token:',
    style={'description_width': '60px'},
    layout=widgets.Layout(width='300px')
)

tg_chat_id_input = widgets.Text(
    placeholder='Telegram Chat ID',
    description='Chat ID:',
    style={'description_width': '60px'},
    layout=widgets.Layout(width='200px')
)

settings_box = widgets.HBox([
    widgets.VBox([
        widgets.HTML('<b style="font-size:14px;">4️⃣ Parameters</b>'),
        widgets.HBox([profit_threshold, check_interval], layout=widgets.Layout(gap='30px'))
    ]),
    widgets.VBox([
        widgets.HTML('<b style="font-size:14px;">5️⃣ Telegram</b>'),
        widgets.HBox([enable_telegram], layout=widgets.Layout(gap='10px')),
        widgets.HBox([tg_token_input, tg_chat_id_input], layout=widgets.Layout(gap='10px'))
    ])
], layout=widgets.Layout(gap='60px', margin='10px 0'))


## ▶️ Run Button
run_button = widgets.Button(
    description='🚀 Start Detection',
    button_style='success',
    layout=widgets.Layout(width='180px', height='40px')
)

button_box = widgets.HBox([run_button], layout=widgets.Layout(justify_content='center', margin='15px 0'))


## 🖥️ Output Areas
table_output = widgets.Output(layout=widgets.Layout(width='100%', border='2px solid #3CB371', padding='15px', height='500px', overflow_y='auto'))
output_area = widgets.Output(layout=widgets.Layout(width='100%', border='2px solid #999', padding='15px', height='1000px', overflow_y='auto'))


# --- Main Layout ---
main_layout = widgets.VBox([
    title,
    widgets.HTML('<hr style="margin: 10px 0; border: 1px solid #ddd;">'),
    row1,
    widgets.HTML('<hr style="margin: 10px 0; border: 1px solid #ddd;">'),
    fee_box,
    widgets.HTML('<hr style="margin: 10px 0; border: 1px solid #ddd;">'),
    settings_box,
    widgets.HTML('<hr style="margin: 10px 0; border: 1px solid #ddd;">'),
    button_box,
    widgets.HTML('<b>📈 Arbitrage Opportunities (Top 10)</b>'),
    table_output,
    widgets.HTML('<b>💻 Log / Console Output</b>'),
    output_area,
], layout=widgets.Layout(width='100%'))


# --- Event Handler Logic ---

def send_telegram_notification(token, chat_id, opportunities, current_time):
    """Sends a single Telegram message containing the timestamp and all opportunities for the batch."""
    if not opportunities:
        return

    if not token or not chat_id:
        print('❌ TG Error: Token or Chat ID is empty')
        return
    
    # Build the message
    msg = f'🕒 <b>Arbitrage Detection Report</b> 🕒\n'
    msg += f'<i>Time: {current_time}</i>\n\n'
    
    # Group opportunities by symbol
    grouped_opportunities = {}
    for opp in opportunities:
        symbol = opp['symbol']
        if symbol not in grouped_opportunities:
            grouped_opportunities[symbol] = []
        grouped_opportunities[symbol].append(opp)
    
    # Sort groups by the highest profit within the group for better display order
    sorted_symbols = sorted(grouped_opportunities.keys(), 
                            key=lambda s: max(opp['profit_percent'] for opp in grouped_opportunities[s]), 
                            reverse=True)

    for symbol in sorted_symbols:
        opps = grouped_opportunities[symbol]
        name = opps[0]['name']
        msg += f'💎 <b>{name} ({symbol}) - {len(opps)} Opportunities</b>\n'
        
        # List all arbitrage opportunities for this coin
        for i, opp in enumerate(opps):
            # Format output
            buy_ex = opp["buy_exchange"].capitalize()
            sell_ex = opp["sell_exchange"].capitalize()
            profit = f'{opp["profit_percent"]:.4f}%'
            buy_price = f'{opp["buy_price"]:.6f}'
            sell_price = f'{opp["sell_price"]:.6f}'

            # Highlight profit and prices
            msg += f'  {i+1}. Net Profit: <b>{profit}</b>\n'
            msg += f'     Buy: {buy_ex} @ <b>${buy_price}</b>\n'
            msg += f'     Sell: {sell_ex} @ <b>${sell_price}</b>\n'
        msg += '\n'

    try:
        url = f'https://api.telegram.org/bot{token}/sendMessage'
        payload = {'chat_id': chat_id, 'text': msg, 'parse_mode': 'HTML'}
        response = requests.post(url, json=payload, timeout=10)
        
        if response.status_code == 200:
            print(f'✅ TG Sent: {len(opportunities)} opportunities in one message')
        else:
            print(f'❌ TG API Error: {response.status_code} - {response.text}')

    except Exception as e:
        print(f'❌ TG Connection Error: {e}')


def get_current_fees():
    return {ex: fee_widgets[ex].value for ex in ALL_EXCHANGES}

running = False
running_lock = threading.Lock()

def on_run_clicked(b):
    global running, detector

    with running_lock:
        if running:
            with output_area: clear_output(); print('⚠️ Detection is already running. Please interrupt the kernel to stop.')
            return
        running = True

    run_button.disabled = True

    exchanges_str = exchange_display.value
    cryptos_symbols_str = crypto_display.value

    exchanges = [e.strip() for e in exchanges_str.split(',') if e.strip() and e.strip() != '(None)']
    selected_symbols = [s.strip() for s in cryptos_symbols_str.split('\n') if s.strip() and s.strip() != '(None)']
    selected_cryptos = [c for c in CRYPTO_PAIRS if c['symbol'] in selected_symbols]


    threshold = profit_threshold.value
    interval = check_interval.value
    enable_tg = enable_telegram.value
    tg_token = tg_token_input.value if enable_tg else ''
    tg_chat_id = tg_chat_id_input.value if enable_tg else ''

    if not exchanges:
        with output_area: clear_output(); print('❌ Select exchanges first')
        with running_lock: running = False; run_button.disabled = False; return
    if not selected_cryptos:
        with output_area: clear_output(); print('❌ Select cryptos first')
        with running_lock: running = False; run_button.disabled = False; return
    if enable_tg and (not tg_token or not tg_chat_id):
        with output_area: clear_output(); print('❌ TG config incomplete (Token or Chat ID missing)')
        with running_lock: running = False; run_button.disabled = False; return

    detector.fees = get_current_fees()
    detector.initialize_exchanges(exchanges)

    with output_area:
        clear_output()
        print('--- Arbitrage Detector Initialized ---')
        print(f'🚨 Stop: interrupt Jupyter kernel')
        print(f'Exchanges: {len(exchanges)} | Cryptos: {len(selected_cryptos)}')
        print(f'Min Profit: {threshold}% | Interval: {interval}s')
        print(f'Telegram: {"Enabled" if enable_tg else "Disabled"}')
        print('------------------------------------')


    iteration = 0
    while True:
        with running_lock:
            if not running: break

        iteration += 1
        round_opportunities = []
        current_batch_time = datetime.now()
        current_batch_time_str = current_batch_time.strftime("%Y-%m-%d %H:%M:%S")

        with output_area:
            clear_output(wait=True)
            print(f'🔄 #{iteration} | {current_batch_time_str} | Fetching Prices via CCXT...')

        max_price_workers = 20 # Limit threads to prevent excessive requests
        with ThreadPoolExecutor(max_workers=max_price_workers) as executor:
            # Submit all crypto price fetching tasks
            futures = {executor.submit(detector.get_crypto_all_prices, crypto, exchanges): crypto for crypto in selected_cryptos}

            for future in as_completed(futures):
                with running_lock:
                    if not running: break

                crypto = futures[future]
                unified_symbol = crypto['symbol']

                try:
                    price_data = future.result()

                    if not price_data or not price_data.get('prices') or len(price_data['prices']) < 2:
                        with output_area: print(f'⚠️ {unified_symbol}: Not enough valid price data received (skipping calculation)')
                        continue

                    log_msg = f"🔎 {unified_symbol} Prices:"
                    for ex, p in price_data['prices'].items():
                        ask = f"{p['ask']:.6f}" if p['ask'] is not None else 'N/A'
                        bid = f"{p['bid']:.6f}" if p['bid'] is not None else 'N/A'
                        log_msg += f" {ex[:4]}={ask}/{bid}"
                    with output_area: print(log_msg)

                    all_pairs = detector.find_all_arbitrage_pairs(price_data)
                    profitable_pairs = [p for p in all_pairs if p['profit_percent'] >= threshold]

                    if profitable_pairs:
                        # Sort by profit for log output and later TG formatting
                        profitable_pairs.sort(key=lambda x: x['profit_percent'], reverse=True)
                        round_opportunities.extend(profitable_pairs)
                        with output_area: print(f'✅ {unified_symbol}: Found {len(profitable_pairs)} opportunities, Max: {profitable_pairs[0]["profit_percent"]:.4f}%')
                    else:
                        with output_area: print(f'ℹ️ {unified_symbol}: No opportunities > {threshold}%')

                except Exception as e:
                    with output_area: print(f'❌ {unified_symbol} General Error during processing: {str(e)}')
            
            with running_lock:
                if not running: break

        # --- TG Notification (One message per batch) ---
        if enable_tg and round_opportunities:
            send_telegram_notification(tg_token, tg_chat_id, round_opportunities, current_batch_time_str)
        # ---------------------------------------------

        if not running: break

        # Display results table
        with table_output:
            clear_output(wait=True)
            if round_opportunities:
                # Use DataFrame for display (Top 10)
                df = pd.DataFrame(round_opportunities).sort_values('profit_percent', ascending=False).head(10)
                
                df_display = df[['symbol', 'buy_exchange', 'sell_exchange', 'buy_price', 'sell_price', 'buy_fee', 'sell_fee', 'profit_percent']].copy()

                df_display.columns = ['Crypto', 'Buy', 'Sell', 'Ask Price (Buy)', 'Bid Price (Sell)', 'Buy Fee (%)', 'Sell Fee (%)', 'Profit (%)']

                # Formatting for display
                df_display['Profit (%)'] = df_display['Profit (%)'].round(4)
                df_display['Buy Fee (%)'] = (df_display['Buy Fee (%)'] * 100).round(4)
                df_display['Sell Fee (%)'] = (df_display['Sell Fee (%)'] * 100).round(4)
                df_display['Ask Price (Buy)'] = df_display['Ask Price (Buy)'].round(6)
                df_display['Bid Price (Sell)'] = df_display['Bid Price (Sell)'].round(6)
                
                df_display['Buy'] = df_display['Buy'].str.capitalize()
                df_display['Sell'] = df_display['Sell'].str.capitalize()

                html_table = df_display.to_html(index=False, classes='table table-striped', float_format='%.6f')
                display(HTML(f'<b>Found Total: {len(round_opportunities)} | Displaying Top {len(df_display)}</b><br>{html_table}'))
            else:
                print(f'ℹ️ {current_batch_time_str}: No opportunities found above the minimum profit threshold.')

        time.sleep(interval)

    with output_area: print('⏹ Stopped.')
    run_button.disabled = False


run_button.on_click(on_run_clicked)
display(main_layout)

## User Guide

1. **Select Exchanges**: Choose exchanges to monitor from the left list (multiple selection)
2. **Select Cryptocurrencies**: Choose cryptos to detect from the left list (multiple selection)
3. **Set Profit Threshold**: Drag the slider to set minimum profit percentage
4. **Configure Telegram (Optional)**:
   - Check "Enable Telegram Push" checkbox
   - Enter your Bot Token and Chat ID
5. **Click "🚀 Start Detection"** to begin monitoring
6. **Click "⏹ Stop"** to end detection

### 💡 How to Get Telegram Configuration
- Send `/newbot` to [@BotFather](https://t.me/BotFather) to create a new Bot
- Copy the Token to the input field above
- Send any message to [@userinfobot](https://t.me/userinfobot) to get your Chat ID

### 📊 Results Interpretation
- **Green Title**: GUI is ready
- **✅ Symbol**: Arbitrage opportunities found
- **ℹ️ Symbol**: No opportunities found yet
- **Profit (%)**: Potential profit percentage
